In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(parent_dir)
print("Current working directory:", os.getcwd())

Current working directory: /local0/home/mfilo/git/CRN-GenerativeAI/applications/molecular_oscillators_3species_reinforce


In [2]:
# Importing packages and modules
from openpyxl import Workbook, load_workbook
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN.iocrns.mass_action_iocrn import MassActionIOCRN
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_from_library import AddReactionFromLibrary
from RL4CRN.rewards.deterministic import oscillation_error

In [3]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Molecular_Oscillators_REINFORCE",        
    workspace="maurice-filo", 
    name=f'Transients_Experiment_{timestamp}',
)
logger = logger.experiment

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/maurice-filo/molecular-oscillators-reinforce/6347fb700112475c80332e2d8d46f2fe



In [4]:
# Construct the template CRN
species_labels = ['X_1', 'X_2', 'X_3']
inputs_labels = ['u_1']
S_R = np.array([[0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=np.int8)
S_P = np.array([[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], dtype=np.int8)
c = np.array([1, 0.1, 0.1, 0.1], dtype=np.float32)
S_I = np.array([[1, 0, 0, 0]], dtype=np.int8)
o = np.array([3], dtype=np.int8)
crn_template = MassActionIOCRN(S_R, S_P, c, S_I, o, species_labels, inputs_labels)
n = len(species_labels)
p = len(inputs_labels)
print('CRN template:')
print(crn_template)

CRN template:
Inputs: ['u_1'] 
Species: ['X_1', 'X_2', 'X_3'] 
Output Species: ['X_3'] 
Reaction 0: 0 -> X_1 ; Rate Constant: 1.0u_1 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.10000000149011612 
Reaction 2: X_2 -> 0 ; Rate Constant: 0.10000000149011612 
Reaction 3: X_3 -> 0 ; Rate Constant: 0.10000000149011612 



In [5]:
# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [6]:
# Flags and filenames
save_flag = False                                               # Save the agent checkpoint
load_flag = False                                               # Load the agent checkpoint 
train_flag = True                                               # Train the agent
save_sheet_flag = True                                          # Save the configuration to an Excel sheet

save_filename = timestamp + '.pth'                              # Filename for saving the agent checkpoint
load_filename = ''                                              # Filename for loading the agent checkpoint
file_name = "Discover_Molecular_Oscillators_REINFORCE.xlsx"           # Filename for saving the Excel sheet

In [7]:
# Hyperparameters
max_added_reactions = 6                             # Maximum number of reactions
N_CPUs = os.cpu_count()                             # Number of CPUs          
N = 30*N_CPUs                                       # Number of samples (batch size)    
width = 1024                                        # Width of the neural networks  
depth = 5                                           # Depth of the neural networks 
deep_layer_size = 1024*10                           # Size of the deep layer encoding the CRNs
allow_input_influence = False                       # Allow input influence in the policy
learning_rate = 1e-5                                # Learning rate for the optimizer 
hall_of_fame_size = 10                              # Size of the hall of fame  
entropy_scheduler = {                               # Entropy scheduler parameters
    'entropy_weight': 0, 
    'entropy_update_coefficient': 0.75, 
    'entropy_schedule': 10000, 
    'minimum_entropy_weight': 0.1
}
risk_scheduler = {                                  # Risk scheduler parameters
    'risk': 0.995, 
    'risk_update': 0.0, 
    'max_risk': 1.0, 
    'risk_schedule': 20
}
epoch_num = 500                                     # Number of epochs for training
render_schedule = 1                                 # Render every # of epochs
mode = {                                            # Mode of the experiment
    'style': 'logger', 
    'task': 'transients', 
    'format': 'image'
}

# Time horizon for the simulation
t_f = 200                                           # Final time for the simulation
N_t = 1000                                          # Number of time steps
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Construct the IOCRN inputs
periods = [10, 15, 20]
u_list = [np.array([1/p]) for p in periods]

# Construct the IOCRN initial conditions
x0_list = [np.array([0, 0, 0], dtype=np.float32)] # list of initial conditions, each initial condition is a numpy array of shape (n,)

# Weight on the damping metric and initial time for peak detection
w = [0/10, 0/10, 10/10]
t0 = 50.

# Construct the compute reward routine
def compute_reward(state):
   f_list = u_list
   return oscillation_error(state, u_list, x0_list, time_horizon, f_list, w, t0, LARGE_NUMBER=1e4)

In [8]:
# Save the experiment discription in an Excel sheet
if save_sheet_flag:
    sheet_name = "Data"
    headers = ["Timestamp", "URL", 
               "Maximum Number of Reactions", "Number of Species", "Number of Inputs", "Batch Size", 
               "Allow Input Influence", 
               "Learning Rate", "Number of Epochs", "Neural Network Depth", "Neural Network Width", "Deep Layer Size", "Number of CPUs",
               "Initial Entropy Weight", "Entropy Update Coefficient", "Entropy Schedule", "Minimum Entropy Weight", 
               "Risk", "Risk Update", "Maximum Risk", "Risk Schedule",
               "Render Schedule", "Hall of Fame Size", 
               "Simulation Time", "Number of Time Steps", "Initial Condition Scenarios", "Input Scenarios"]
    data_row = [timestamp, logger.url, 
                max_added_reactions, n, p, N,
                allow_input_influence, 
                learning_rate, epoch_num, depth, width, deep_layer_size, N_CPUs, 
                entropy_scheduler['entropy_weight'], entropy_scheduler['entropy_update_coefficient'], entropy_scheduler['entropy_schedule'], entropy_scheduler['minimum_entropy_weight'],
                risk_scheduler['risk'], risk_scheduler['risk_update'], risk_scheduler['max_risk'], risk_scheduler['risk_schedule'],
                render_schedule, hall_of_fame_size,
                t_f, N_t, len(x0_list), len(u_list)]

    if os.path.exists(file_name):
        wb = load_workbook(file_name)
        if sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
        else:
            ws = wb.create_sheet(sheet_name)
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = sheet_name

    if ws.max_column < 2:
        for i, header in enumerate(headers, start=1):
            ws.cell(row=i, column=1, value=header)
    next_col = ws.max_column + 1
    for i, value in enumerate(data_row, start=1):
        ws.cell(row=i, column=next_col, value=value)

    wb.save(file_name)
    print(f"New experiment data saved in column {next_col} of '{file_name}'.")

New experiment data saved in column 6 of 'Discover_Molecular_Oscillators_REINFORCE.xlsx'.


In [9]:
# Construct parallel environments
crn_0 = crn_template.clone()
mult_env = ParallelEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, N_CPUs=N_CPUs, logger=logger)
# mult_env = SerialEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, logger=logger)

In [10]:
# Construct the Agent
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_head_attributes = {"hidden_size": width, "num_layers": depth}
rate_head_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_head_attributes = {"hidden_size": width, "num_layers": depth}
output_head_attributes = {"hidden_size": width, "num_layers": depth}
M = crn_0.get_reactions_range()
policy = AddReactionFromLibrary(M, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, continuous_distribution='lognormal', allow_input_influence=False, device=device)
agent = REINFORCEAgent(policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_scheduler=entropy_scheduler, risk_scheduler=risk_scheduler, device=device)
if load_flag:
    agent.policy.load_state_dict(torch.load(save_filename, map_location=device)) 

In [ ]:
# Training Loop
if train_flag:
    for i in tqdm(range(epoch_num)):
        mult_env.reset()
        for j in range(max_added_reactions + mult_env.envs[0].crn_template.num_unknown_rates):
            states = mult_env.observe()
            actions = agent.act(states)
            out = mult_env.step(actions, mode='reaction index')
        rewards = mult_env.get_reward(compute_reward)
        agent.update(rewards, step_iteration=i)
        if i % render_schedule == 0:
            mult_env.render(rewards, n_best=1, disregarded_percentage=0.9, mode=mode)

 63%|██████▎   | 315/500 [48:15<27:43,  8.99s/it]  

In [ ]:
# Test the model
n_plot = 5
mult_env.reset()
for j in range(max_added_reactions + mult_env.envs[0].crn_template.num_unknown_rates):
    states = mult_env.observe()
    actions = agent.act(states)
    out = mult_env.step(actions, mode='reaction index')
rewards = mult_env.get_reward(compute_reward)

# Gather the CRNs from the environments
crns = mult_env.gather()

# Sort the CRNs by rewards
sorted_crns_rewards = sorted(zip(crns, rewards), key=lambda x: x[1])

# Plot the transient responses
for i in range(n_plot):
    time_horizon, x_list, y_list, last_task_info = sorted_crns_rewards[i][0].transient_response(u_list, x0_list, time_horizon)
    sorted_crns_rewards[i][0].plot_transient_response()
    print(f"IOCRN {i}, Reward: {sorted_crns_rewards[i][1]}")
    print(sorted_crns_rewards[0][0])